In [3]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "execution_count": 12,
      "metadata": {
        "id": "4-2PWHTQaQyt"
      },
      "outputs": [],
      "source": [
        "!pip install -q pyspark\n",
        "\n",
        "import os, shutil, glob, hashlib\n",
        "os.makedirs(\"data_lake/bronze\", exist_ok=True)\n",
        "os.makedirs(\"data_lake/silver/preprocessed_sales\", exist_ok=True)"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from google.colab import files\n",
        "files.upload()\n",
        "shutil.move(\"sales.csv\", \"data_lake/bronze/sales.csv\")\n",
        "\n",
        "md5 = lambda p: hashlib.md5(open(p, \"rb\").read()).hexdigest()\n",
        "bronze_md5 = md5(\"data_lake/bronze/sales.csv\")   # used at the end to prove Bronze is unchanged"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 74
        },
        "id": "lq9P5to9aiKV",
        "outputId": "6df9066d-a4d2-4292-f71c-27864bf025a4"
      },
      "execution_count": 13,
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "text/plain": [
              "<IPython.core.display.HTML object>"
            ],
            "text/html": [
              "\n",
              "     <input type=\"file\" id=\"files-a60899c1-d25e-4fca-af34-f05c8ec07faf\" name=\"files[]\" multiple disabled\n",
              "        style=\"border:none\" />\n",
              "     <output id=\"result-a60899c1-d25e-4fca-af34-f05c8ec07faf\">\n",
              "      Upload widget is only available when the cell has been executed in the\n",
              "      current browser session. Please rerun this cell to enable.\n",
              "      </output>\n",
              "      <script>// Copyright 2017 Google LLC\n",
              "//\n",
              "// Licensed under the Apache License, Version 2.0 (the \"License\");\n",
              "// you may not use this file except in compliance with the License.\n",
              "// You may obtain a copy of the License at\n",
              "//\n",
              "//      http://www.apache.org/licenses/LICENSE-2.0\n",
              "//\n",
              "// Unless required by applicable law or agreed to in writing, software\n",
              "// distributed under the License is distributed on an \"AS IS\" BASIS,\n",
              "// WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n",
              "// See the License for the specific language governing permissions and\n",
              "// limitations under the License.\n",
              "\n",
              "/**\n",
              " * @fileoverview Helpers for google.colab Python module.\n",
              " */\n",
              "(function(scope) {\n",
              "function span(text, styleAttributes = {}) {\n",
              "  const element = document.createElement('span');\n",
              "  element.textContent = text;\n",
              "  for (const key of Object.keys(styleAttributes)) {\n",
              "    element.style[key] = styleAttributes[key];\n",
              "  }\n",
              "  return element;\n",
              "}\n",
              "\n",
              "// Max number of bytes which will be uploaded at a time.\n",
              "const MAX_PAYLOAD_SIZE = 100 * 1024;\n",
              "\n",
              "function _uploadFiles(inputId, outputId) {\n",
              "  const steps = uploadFilesStep(inputId, outputId);\n",
              "  const outputElement = document.getElementById(outputId);\n",
              "  // Cache steps on the outputElement to make it available for the next call\n",
              "  // to uploadFilesContinue from Python.\n",
              "  outputElement.steps = steps;\n",
              "\n",
              "  return _uploadFilesContinue(outputId);\n",
              "}\n",
              "\n",
              "// This is roughly an async generator (not supported in the browser yet),\n",
              "// where there are multiple asynchronous steps and the Python side is going\n",
              "// to poll for completion of each step.\n",
              "// This uses a Promise to block the python side on completion of each step,\n",
              "// then passes the result of the previous step as the input to the next step.\n",
              "function _uploadFilesContinue(outputId) {\n",
              "  const outputElement = document.getElementById(outputId);\n",
              "  const steps = outputElement.steps;\n",
              "\n",
              "  const next = steps.next(outputElement.lastPromiseValue);\n",
              "  return Promise.resolve(next.value.promise).then((value) => {\n",
              "    // Cache the last promise value to make it available to the next\n",
              "    // step of the generator.\n",
              "    outputElement.lastPromiseValue = value;\n",
              "    return next.value.response;\n",
              "  });\n",
              "}\n",
              "\n",
              "/**\n",
              " * Generator function which is called between each async step of the upload\n",
              " * process.\n",
              " * @param {string} inputId Element ID of the input file picker element.\n",
              " * @param {string} outputId Element ID of the output display.\n",
              " * @return {!Iterable<!Object>} Iterable of next steps.\n",
              " */\n",
              "function* uploadFilesStep(inputId, outputId) {\n",
              "  const inputElement = document.getElementById(inputId);\n",
              "  inputElement.disabled = false;\n",
              "\n",
              "  const outputElement = document.getElementById(outputId);\n",
              "  outputElement.innerHTML = '';\n",
              "\n",
              "  const pickedPromise = new Promise((resolve) => {\n",
              "    inputElement.addEventListener('change', (e) => {\n",
              "      resolve(e.target.files);\n",
              "    });\n",
              "  });\n",
              "\n",
              "  const cancel = document.createElement('button');\n",
              "  inputElement.parentElement.appendChild(cancel);\n",
              "  cancel.textContent = 'Cancel upload';\n",
              "  const cancelPromise = new Promise((resolve) => {\n",
              "    cancel.onclick = () => {\n",
              "      resolve(null);\n",
              "    };\n",
              "  });\n",
              "\n",
              "  // Wait for the user to pick the files.\n",
              "  const files = yield {\n",
              "    promise: Promise.race([pickedPromise, cancelPromise]),\n",
              "    response: {\n",
              "      action: 'starting',\n",
              "    }\n",
              "  };\n",
              "\n",
              "  cancel.remove();\n",
              "\n",
              "  // Disable the input element since further picks are not allowed.\n",
              "  inputElement.disabled = true;\n",
              "\n",
              "  if (!files) {\n",
              "    return {\n",
              "      response: {\n",
              "        action: 'complete',\n",
              "      }\n",
              "    };\n",
              "  }\n",
              "\n",
              "  for (const file of files) {\n",
              "    const li = document.createElement('li');\n",
              "    li.append(span(file.name, {fontWeight: 'bold'}));\n",
              "    li.append(span(\n",
              "        `(${file.type || 'n/a'}) - ${file.size} bytes, ` +\n",
              "        `last modified: ${\n",
              "            file.lastModifiedDate ? file.lastModifiedDate.toLocaleDateString() :\n",
              "                                    'n/a'} - `));\n",
              "    const percent = span('0% done');\n",
              "    li.appendChild(percent);\n",
              "\n",
              "    outputElement.appendChild(li);\n",
              "\n",
              "    const fileDataPromise = new Promise((resolve) => {\n",
              "      const reader = new FileReader();\n",
              "      reader.onload = (e) => {\n",
              "        resolve(e.target.result);\n",
              "      };\n",
              "      reader.readAsArrayBuffer(file);\n",
              "    });\n",
              "    // Wait for the data to be ready.\n",
              "    let fileData = yield {\n",
              "      promise: fileDataPromise,\n",
              "      response: {\n",
              "        action: 'continue',\n",
              "      }\n",
              "    };\n",
              "\n",
              "    // Use a chunked sending to avoid message size limits. See b/62115660.\n",
              "    let position = 0;\n",
              "    do {\n",
              "      const length = Math.min(fileData.byteLength - position, MAX_PAYLOAD_SIZE);\n",
              "      const chunk = new Uint8Array(fileData, position, length);\n",
              "      position += length;\n",
              "\n",
              "      const base64 = btoa(String.fromCharCode.apply(null, chunk));\n",
              "      yield {\n",
              "        response: {\n",
              "          action: 'append',\n",
              "          file: file.name,\n",
              "          data: base64,\n",
              "        },\n",
              "      };\n",
              "\n",
              "      let percentDone = fileData.byteLength === 0 ?\n",
              "          100 :\n",
              "          Math.round((position / fileData.byteLength) * 100);\n",
              "      percent.textContent = `${percentDone}% done`;\n",
              "\n",
              "    } while (position < fileData.byteLength);\n",
              "  }\n",
              "\n",
              "  // All done.\n",
              "  yield {\n",
              "    response: {\n",
              "      action: 'complete',\n",
              "    }\n",
              "  };\n",
              "}\n",
              "\n",
              "scope.google = scope.google || {};\n",
              "scope.google.colab = scope.google.colab || {};\n",
              "scope.google.colab._files = {\n",
              "  _uploadFiles,\n",
              "  _uploadFilesContinue,\n",
              "};\n",
              "})(self);\n",
              "</script> "
            ]
          },
          "metadata": {}
        },
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Saving sales.csv to sales.csv\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql import SparkSession, functions as F\n",
        "spark = SparkSession.builder.appName(\"DataLake\").getOrCreate()\n",
        "spark.conf.set(\"spark.sql.ansi.enabled\", \"false\")\n",
        "spark.conf.set(\"spark.sql.legacy.timeParserPolicy\", \"CORRECTED\")"
      ],
      "metadata": {
        "id": "CF7tAVAPapxt"
      },
      "execution_count": 14,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "raw = spark.read.option(\"header\", True).csv(\"data_lake/bronze/sales.csv\")\n",
        "raw.printSchema()\n",
        "raw.show(5, truncate=False)\n",
        "rows_before = raw.count()\n",
        "print(\"Total records:\", rows_before)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "JQDasYfBawLI",
        "outputId": "ec1402a2-891c-4969-b224-7f0b7a444c3f"
      },
      "execution_count": 15,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "root\n",
            " |-- order_id: string (nullable = true)\n",
            " |-- customer_id: string (nullable = true)\n",
            " |-- customer_name: string (nullable = true)\n",
            " |-- product: string (nullable = true)\n",
            " |-- category: string (nullable = true)\n",
            " |-- quantity: string (nullable = true)\n",
            " |-- unit_price: string (nullable = true)\n",
            " |-- discount_percent: string (nullable = true)\n",
            " |-- payment_method: string (nullable = true)\n",
            " |-- city: string (nullable = true)\n",
            " |-- state: string (nullable = true)\n",
            " |-- order_date: string (nullable = true)\n",
            " |-- order_status: string (nullable = true)\n",
            "\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |\n",
            "|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |\n",
            "|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |\n",
            "|100004  |C0051      |Tanvi Sharma  |Running Shoes|Sports  |6       |2260      |35              |Net Banking   |Lucknow   |Uttar Pradesh |2026-07-31|Cancelled   |\n",
            "|100005  |C0227      |Aarav Kumar   |Coffee 500g  |Grocery |10      |740       |12              |Debit Card    |Vijayawada|Andhra Pradesh|2026-08-30|Delivered   |\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "only showing top 5 rows\n",
            "Total records: 1000\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# Duplicates\n",
        "print(\"Exact duplicate rows:\", rows_before - raw.dropDuplicates().count())\n",
        "\n",
        "# Nulls / blanks\n",
        "raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == \"\"), 1).otherwise(0)).alias(c)\n",
        "            for c in raw.columns]).show()\n",
        "\n",
        "# Text problems\n",
        "text_cols = [\"customer_name\",\"product\",\"category\",\"city\",\"state\",\"payment_method\",\"order_status\"]\n",
        "for c in text_cols:\n",
        "    spaces = raw.filter(F.col(c) != F.regexp_replace(F.trim(F.col(c)), r\"\\s+\", \" \")).count()\n",
        "    variants = raw.select(F.trim(F.col(c))).distinct().count() - raw.select(F.lower(F.trim(F.col(c)))).distinct().count()\n",
        "    print(f\"{c:15} extra-space rows: {spaces:3} | case variants: {variants}\")\n",
        "\n",
        "# Invalid numbers\n",
        "num = (raw.withColumn(\"q\", F.col(\"quantity\").cast(\"double\"))\n",
        "          .withColumn(\"p\", F.col(\"unit_price\").cast(\"double\"))\n",
        "          .withColumn(\"d\", F.col(\"discount_percent\").cast(\"double\")))\n",
        "print(\"quantity <= 0:\", num.filter(F.col(\"q\") <= 0).count())\n",
        "print(\"unit_price <= 0:\", num.filter(F.col(\"p\") <= 0).count())\n",
        "print(\"discount <0 or >100:\", num.filter((F.col(\"d\") < 0) | (F.col(\"d\") > 100)).count())\n",
        "\n",
        "# Dates\n",
        "formats = [\"yyyy-MM-dd\", \"yyyy/MM/dd\", \"dd/MM/yyyy\", \"dd-MM-yyyy\", \"MM-dd-yyyy\"]\n",
        "def parse_date(c):\n",
        "    return F.coalesce(*[F.to_date(F.trim(F.col(c)), f) for f in formats])\n",
        "\n",
        "raw.withColumn(\"pattern\", F.regexp_replace(F.regexp_replace(\"order_date\", r\"\\d\", \"9\"), r\"[A-Za-z]\", \"a\")) \\\n",
        "   .groupBy(\"pattern\").count().show()                      # shows mixed date formats\n",
        "bad = raw.filter(F.col(\"order_date\").isNotNull() & parse_date(\"order_date\").isNull())\n",
        "print(\"Invalid dates:\", bad.count())\n",
        "bad.select(\"order_id\", \"order_date\").show()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "hx2eDsNPa2sN",
        "outputId": "c77103db-a8ff-4c7c-8a0c-b8492a2f4feb"
      },
      "execution_count": 16,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Exact duplicate rows: 10\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n",
            "customer_name   extra-space rows:   3 | case variants: 0\n",
            "product         extra-space rows:   0 | case variants: 0\n",
            "category        extra-space rows:   0 | case variants: 5\n",
            "city            extra-space rows:   3 | case variants: 5\n",
            "state           extra-space rows:   0 | case variants: 0\n",
            "payment_method  extra-space rows:   0 | case variants: 0\n",
            "order_status    extra-space rows:   0 | case variants: 0\n",
            "quantity <= 0: 12\n",
            "unit_price <= 0: 8\n",
            "discount <0 or >100: 8\n",
            "+----------+-----+\n",
            "|   pattern|count|\n",
            "+----------+-----+\n",
            "|99-99-9999|    1|\n",
            "|aaa-a-aaaa|    1|\n",
            "|99/99/9999|    1|\n",
            "|9999-99-99|  996|\n",
            "|9999/99/99|    1|\n",
            "+----------+-----+\n",
            "\n",
            "Invalid dates: 4\n",
            "+--------+----------+\n",
            "|order_id|order_date|\n",
            "+--------+----------+\n",
            "|  100097|31/02/2026|\n",
            "|  100448|not-a-date|\n",
            "|  100634|2025-02-30|\n",
            "|  100637|2026-13-05|\n",
            "+--------+----------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# 1. Remove exact duplicates\n",
        "df = raw.dropDuplicates()\n",
        "\n",
        "# 3. Trim, collapse spaces, blank -> null\n",
        "for c in df.columns:\n",
        "    df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r\"\\s+\", \" \"))\n",
        "    df = df.withColumn(c, F.when(F.col(c) == \"\", None).otherwise(F.col(c)))\n",
        "\n",
        "# 3. Standardize capitalization (not on product, because it would turn \"5L\" into \"5l\")\n",
        "for c in [\"customer_name\", \"category\", \"city\"]:\n",
        "    df = df.withColumn(c, F.initcap(F.col(c)))\n",
        "\n",
        "# 4. Cast numbers, then turn invalid values into null (the row is kept)\n",
        "df = (df.withColumn(\"quantity\", F.col(\"quantity\").cast(\"int\"))\n",
        "        .withColumn(\"unit_price\", F.col(\"unit_price\").cast(\"double\"))\n",
        "        .withColumn(\"discount_percent\", F.col(\"discount_percent\").cast(\"double\")))\n",
        "df = (df.withColumn(\"quantity\", F.when(F.col(\"quantity\") > 0, F.col(\"quantity\")))\n",
        "        .withColumn(\"unit_price\", F.when(F.col(\"unit_price\") > 0, F.col(\"unit_price\")))\n",
        "        .withColumn(\"discount_percent\", F.when(F.col(\"discount_percent\").between(0, 100), F.col(\"discount_percent\"))))\n",
        "\n",
        "# 5. Dates -> one format (yyyy-MM-dd); invalid dates become null\n",
        "df = df.withColumn(\"order_date\", parse_date(\"order_date\"))\n",
        "\n",
        "# 2. Missing values\n",
        "qty_median   = df.approxQuantile(\"quantity\", [0.5], 0.01)[0]\n",
        "price_median = df.approxQuantile(\"unit_price\", [0.5], 0.01)[0]\n",
        "prod_med = df.groupBy(\"product\").agg(F.percentile_approx(\"unit_price\", 0.5).alias(\"prod_price\"))\n",
        "df = (df.join(prod_med, \"product\", \"left\")\n",
        "        .withColumn(\"unit_price\", F.coalesce(\"unit_price\", \"prod_price\", F.lit(price_median)))\n",
        "        .drop(\"prod_price\"))\n",
        "df = df.fillna({\"quantity\": int(qty_median), \"discount_percent\": 0.0,\n",
        "                \"customer_name\": \"Unknown\", \"payment_method\": \"Unknown\", \"city\": \"Unknown\"})\n",
        "\n",
        "silver = df.select(raw.columns).orderBy(\"order_id\")\n",
        "rows_after = silver.count()"
      ],
      "metadata": {
        "id": "WO-vJsugbXKY"
      },
      "execution_count": 17,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "out = \"data_lake/silver/preprocessed_sales\"\n",
        "silver.coalesce(1).write.mode(\"overwrite\").option(\"header\", True).csv(out)\n",
        "shutil.copy(glob.glob(f\"{out}/part-*.csv\")[0], \"data_lake/silver/sales_silver.csv\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 35
        },
        "id": "cTF2S69ybdw_",
        "outputId": "2af45b90-7b5f-4e40-c693-116d5034d627"
      },
      "execution_count": 18,
      "outputs": [
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "'data_lake/silver/sales_silver.csv'"
            ],
            "application/vnd.google.colaboratory.intrinsic+json": {
              "type": "string"
            }
          },
          "metadata": {},
          "execution_count": 18
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Rows before:\", rows_before, \"| Rows after:\", rows_after)   # expect 1000 -> 990\n",
        "spark.read.option(\"header\", True).csv(out).show(5)\n",
        "\n",
        "silver.select([F.sum(F.col(c).isNull().cast(\"int\")).alias(c) for c in silver.columns]).show()\n",
        "print(\"Bronze unchanged:\", bronze_md5 == md5(\"data_lake/bronze/sales.csv\"))   # must be True"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "IOJ0kMUobjX1",
        "outputId": "5141c40f-6c5f-4aff-fd72-f028772cfb81"
      },
      "execution_count": 19,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Rows before: 1000 | Rows after: 990\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|    6710.0|            20.0|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|\n",
            "|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|     680.0|             5.0|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|\n",
            "|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|    3080.0|            12.0|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|\n",
            "|  100004|      C0051|  Tanvi Sharma|Running Shoes|  Sports|       6|    2260.0|            35.0|   Net Banking|   Lucknow| Uttar Pradesh|2026-07-31|   Cancelled|\n",
            "|  100005|      C0227|   Aarav Kumar|  Coffee 500g| Grocery|      10|     740.0|            12.0|    Debit Card|Vijayawada|Andhra Pradesh|2026-08-30|   Delivered|\n",
            "+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "only showing top 5 rows\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         4|           0|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n",
            "Bronze unchanged: True\n"
          ]
        }
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Preprocessing summary\n",
        "- Removed exact duplicate rows\n",
        "- Trimmed spaces and standardized capitalization (customer_name, category, city)\n",
        "- Converted blank values to null and filled them (median for quantity/unit_price, 0 for discount, \"Unknown\" for text)\n",
        "- Validated quantity, unit_price and discount_percent; invalid values were nulled and then imputed\n",
        "- Converted all valid dates to yyyy-MM-dd; invalid dates were set to null"
      ],
      "metadata": {
        "id": "mgjFiv-bbtBC"
      }
    },
    {
      "cell_type": "code",
      "source": [
        "files.download(\"data_lake/silver/sales_silver.csv\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 17
        },
        "id": "p-XCO7GxcZID",
        "outputId": "8bfa57de-cc94-4ffc-96e5-20416652131f"
      },
      "execution_count": 20,
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "text/plain": [
              "<IPython.core.display.Javascript object>"
            ],
            "application/javascript": [
              "\n",
              "    async function download(id, filename, size) {\n",
              "      if (!google.colab.kernel.accessAllowed) {\n",
              "        return;\n",
              "      }\n",
              "      const div = document.createElement('div');\n",
              "      const label = document.createElement('label');\n",
              "      label.textContent = `Downloading \"${filename}\": `;\n",
              "      div.appendChild(label);\n",
              "      const progress = document.createElement('progress');\n",
              "      progress.max = size;\n",
              "      div.appendChild(progress);\n",
              "      document.body.appendChild(div);\n",
              "\n",
              "      const buffers = [];\n",
              "      let downloaded = 0;\n",
              "\n",
              "      const channel = await google.colab.kernel.comms.open(id);\n",
              "      // Send a message to notify the kernel that we're ready.\n",
              "      channel.send({})\n",
              "\n",
              "      for await (const message of channel.messages) {\n",
              "        // Send a message to notify the kernel that we're ready.\n",
              "        channel.send({})\n",
              "        if (message.buffers) {\n",
              "          for (const buffer of message.buffers) {\n",
              "            buffers.push(buffer);\n",
              "            downloaded += buffer.byteLength;\n",
              "            progress.value = downloaded;\n",
              "          }\n",
              "        }\n",
              "      }\n",
              "      const blob = new Blob(buffers, {type: 'application/binary'});\n",
              "      const a = document.createElement('a');\n",
              "      a.href = window.URL.createObjectURL(blob);\n",
              "      a.download = filename;\n",
              "      div.appendChild(a);\n",
              "      a.click();\n",
              "      div.remove();\n",
              "    }\n",
              "  "
            ]
          },
          "metadata": {}
        },
        {
          "output_type": "display_data",
          "data": {
            "text/plain": [
              "<IPython.core.display.Javascript object>"
            ],
            "application/javascript": [
              "download(\"download_0bbc458a-e3bd-44b1-bb15-df45c842a24b\", \"sales_silver.csv\", 111510)"
            ]
          },
          "metadata": {}
        }
      ]
    }
  ]
}

{'nbformat': 4,
 'nbformat_minor': 0,
 'metadata': {'colab': {'provenance': []},
  'kernelspec': {'name': 'python3', 'display_name': 'Python 3'},
  'language_info': {'name': 'python'}},
 'cells': [{'cell_type': 'code',
   'execution_count': 12,
   'metadata': {'id': '4-2PWHTQaQyt'},
   'outputs': [],
   'source': ['!pip install -q pyspark\n',
    '\n',
    'import os, shutil, glob, hashlib\n',
    'os.makedirs("data_lake/bronze", exist_ok=True)\n',
    'os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)']},
  {'cell_type': 'code',
   'source': ['from google.colab import files\n',
    'files.upload()\n',
    'shutil.move("sales.csv", "data_lake/bronze/sales.csv")\n',
    '\n',
    'md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()\n',
    'bronze_md5 = md5("data_lake/bronze/sales.csv")   # used at the end to prove Bronze is unchanged'],
   'metadata': {'colab': {'base_uri': 'https://localhost:8080/', 'height': 74},
    'id': 'lq9P5to9aiKV',
    'outputId': '6d